# ICD-10 Complete Pipeline Lab

This notebook runs the whole project workflow from raw data to Kaggle-ready submissions:

1. Load raw Kaggle data
2. Clean and normalize text
3. Build exact and fuzzy matching baselines
4. Build TF-IDF retrieval over ICD descriptions
5. Train classical ML models
6. Optionally test embedding and transformer models
7. Combine model outputs into ensembles
8. Save final submission CSVs

The default run avoids the slowest deep-learning cells. Set the flags in the setup cell if you want to run them.

In [15]:
from pathlib import Path
import re
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from scipy.special import expit
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PREDICTIONS_DIR = PROJECT_ROOT / 'output' / 'predictions'
SUBMISSIONS_DIR = PROJECT_ROOT / 'output' / 'submissions'
MODELS_DIR = PROJECT_ROOT / 'models'

for path in [DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Keep these false unless you explicitly want the slow experiments.
RUN_SENTENCE_EMBEDDINGS = False
RUN_TRANSFORMER_FINETUNE = False

RANDOM_STATE = 42
print(PROJECT_ROOT)

d:\Users\herme\UNI\2\Natural language\project\project


## 1. Load Raw Data

In [16]:
train_raw = pd.read_csv(DATA_RAW_DIR / 'codification_data.csv')
test_raw = pd.read_csv(DATA_RAW_DIR / 'leaderboard_data.csv')
reference_raw = pd.read_csv(DATA_RAW_DIR / 'icd_d_p_pairs.csv')

print(f'train:     {train_raw.shape}')
print(f'test:      {test_raw.shape}')
print(f'reference: {reference_raw.shape}')
display(train_raw.head())
display(test_raw.head())
display(reference_raw.head())

train:     (13700, 2)
test:      (6667, 2)
reference: (179742, 3)


,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


,id,Literal
0,1,AMNIODRENAJE
1,2,Hiperparatiroidismo primario
2,3,MIGRANYA parto
3,4,VHC
4,5,Absceso mama izq


,Code,D_P,Description
0,A00,D,Cólera
1,A000,D,"Cólera debido a Vibrio cholerae 01, biotipo ch..."
2,A001,D,"Cólera debido a Vibrio cholerae 01, biotipo El..."
3,A009,D,"Cólera, no especificado"
4,A01,D,Fiebres tifoidea y paratifoidea


## 2. Preprocess Data

In [17]:
def basic_clean(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('`', "'").replace('Â´', "'")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_text(text):
    # Lowercase and no accents
    text = basic_clean(text).lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text.strip()


train = train_raw.copy()
test = test_raw.copy()
reference = reference_raw.copy()

# Normalize
train['Literal_clean'] = train['Literal'].apply(basic_clean)
train['Literal_norm'] = train['Literal'].apply(normalize_text)
test['Literal_clean'] = test['Literal'].apply(basic_clean)
test['Literal_norm'] = test['Literal'].apply(normalize_text)
reference['Description_norm'] = reference['Description'].apply(normalize_text)

# ICD diagnosis codes start with a letter; procedure codes in this dataset start with a digit.
train['code_type'] = train['Code'].astype(str).str[0].apply(lambda c: 'diagnosis' if c.isalpha() else 'procedure')

# The Kaggle category target is the first character of the full ICD code.
train['chapter'] = train['Code'].astype(str).str[0]
reference['chapter'] = reference['Code'].astype(str).str[0]

# Mark which training codes are present in the official ICD reference table.
train['is_icd10'] = train['Code'].isin(set(reference['Code']))

# Add ICD metadata to the training rows when the code is found in the reference table.
train = train.merge(reference[['Code', 'D_P', 'Description']], on='Code', how='left')

# Save to csv
train.to_csv(DATA_PROCESSED_DIR / 'train_preprocessed.csv', index=False)
test.to_csv(DATA_PROCESSED_DIR / 'test_preprocessed.csv', index=False)
reference.to_csv(DATA_PROCESSED_DIR / 'reference_preprocessed.csv', index=False)

print(train.shape, test.shape, reference.shape)
print(train['chapter'].value_counts().head(15))

(13700, 9) (6667, 4) (179742, 5)
chapter
Z    1715
O    1505
0    1141
6     637
3     592
B     579
N     536
E     500
V     491
5     408
8     406
2     404
4     399
7     395
K     381
Name: count, dtype: int64


## 3. Exact And Fuzzy Matching

In [18]:
def build_lookup(df, label_col='Code'):
    lookup = {}
    for literal_norm, group in df.groupby('Literal_norm'):
        lookup[literal_norm] = group[label_col].value_counts().index[0]
    return lookup


def exact_fuzzy_predict(train_df, predict_df, label_col='Code', threshold=60):
    # First try exact normalized-literal lookup, then fuzzy match unresolved rows.
    lookup = build_lookup(train_df, label_col)
    out = predict_df.copy()
    out['exact_pred'] = out['Literal_norm'].map(lookup)
    out['fuzzy_pred'] = out['exact_pred']
    out['fuzzy_score'] = out['exact_pred'].apply(lambda x: 100.0 if pd.notna(x) else 0.0)
    out['match_method'] = out['exact_pred'].apply(lambda x: 'exact' if pd.notna(x) else 'unresolved')

    keys = list(lookup.keys())
    for idx, row in out[out['exact_pred'].isna()].iterrows():
        result = process.extractOne(row['Literal_norm'], keys, scorer=fuzz.ratio, score_cutoff=threshold)
        if result is not None:
            out.loc[idx, 'fuzzy_pred'] = lookup[result[0]]
            out.loc[idx, 'fuzzy_score'] = result[1]
            out.loc[idx, 'match_method'] = 'fuzzy'
    return out


train_split, val_split = train_test_split(train, test_size=0.2, random_state=RANDOM_STATE)
val_match = exact_fuzzy_predict(train_split, val_split, label_col='chapter', threshold=60)
val_match['match_correct'] = val_match['fuzzy_pred'] == val_match['chapter']
print(f"chapter exact/fuzzy validation: {val_match['match_correct'].mean():.4f}")
print(val_match['match_method'].value_counts())

test_match = exact_fuzzy_predict(train, test, label_col='Code', threshold=60)
step1 = test_match[['id', 'Literal', 'Literal_clean', 'Literal_norm', 'fuzzy_pred', 'match_method']].copy()
step1 = step1.rename(columns={'fuzzy_pred': 'step1_code', 'match_method': 'step1_method'})
step1.to_csv(PREDICTIONS_DIR / 'notebook_step1_predictions.csv', index=False)
step1.head()

chapter exact/fuzzy validation: 0.5047
match_method
exact         1474
fuzzy         1245
unresolved      21
Name: count, dtype: int64


,id,Literal,Literal_clean,Literal_norm,step1_code,step1_method
0,1,AMNIODRENAJE,AMNIODRENAJE,amniodrenaje,10903ZU,exact
1,2,Hiperparatiroidismo primario,Hiperparatiroidismo primario,hiperparatiroidismo primario,E210,exact
2,3,MIGRANYA parto,MIGRANYA parto,migranya parto,G43909,fuzzy
3,4,VHC,VHC,vhc,B182,exact
4,5,Absceso mama izq,Absceso mama izq,absceso mama izq,6110,fuzzy


## 4. TF-IDF Retrieval Over ICD Descriptions

Compares each literal to the official ICD descriptions and retrieves the closest code description. It helps most when a test literal does not appear in the training data but resembles an ICD description or a rare code description. Weaker for abbreviations and very short or noisy text.

In [19]:
def build_retrieval_index(reference_df):
    char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
    word_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    char_ref = char_vec.fit_transform(reference_df['Description_norm'].fillna(''))
    word_ref = word_vec.fit_transform(reference_df['Description_norm'].fillna(''))
    return char_vec, word_vec, char_ref, word_ref


def retrieve_best(df, reference_df, index, batch_size=100):
    char_vec, word_vec, char_ref, word_ref = index
    texts = df['Literal_norm'].fillna('')
    x_char = char_vec.transform(texts)
    x_word = word_vec.transform(texts)
    codes, chapters, scores = [], [], []
    # Compute similarities in batches so the reference matrix does not blow up memory.
    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))
        combined = (cosine_similarity(x_char[start:end], char_ref) + cosine_similarity(x_word[start:end], word_ref)) / 2
        best = combined.argmax(axis=1)
        for j in range(end - start):
            idx = best[j]
            code = reference_df.iloc[idx]['Code']
            codes.append(code)
            chapters.append(str(code)[0])
            scores.append(float(combined[j, idx]))
        del combined
    out = df.copy()
    out['retrieval_code'] = codes
    out['retrieval_chapter'] = chapters
    out['retrieval_score'] = scores
    return out


start = time.time()
retrieval_index = build_retrieval_index(reference)
val_ret = retrieve_best(val_split, reference, retrieval_index)
print(f"retrieval chapter validation: {(val_ret['retrieval_chapter'] == val_ret['chapter']).mean():.4f}")
print(f'done in {time.time() - start:.1f}s')

retrieval chapter validation: 0.2967
done in 53.9s


## 5. Classical ML Chapter Models

These models treat chapter prediction as a supervised text classification problem. Each literal is converted into sparse TF-IDF features built from character n-grams and word n-grams.

- **`svc_plain`**: a baseline linear Support Vector Classifier using moderate character and word n-gram ranges. It learns one-vs-rest separating hyperplanes between chapters. This is a strong default for sparse TF-IDF text because it focuses on decision boundaries rather than probability estimates.

- **`svc_tuned`**: a slightly more expressive LinearSVC variant with wider n-gram ranges and `min_df=1`, so it keeps rarer fragments. This can help with uncommon abbreviations, rare terms, and small spelling variants, but it may also pick up more noise.

- **`logreg`**: a logistic regression classifier over the same kind of TF-IDF features. It is usually a little smoother than LinearSVC and provides a different decision pattern, which makes it useful as an ensemble member even when it is not the single best model.

In [20]:
def build_text_features(char_range=(3, 5), word_range=(1, 2), min_df=2):
    return FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=char_range, min_df=min_df, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word', ngram_range=word_range, min_df=min_df, sublinear_tf=True)),
    ])


chapter_models = {}

models_to_try = {
    'svc_plain': Pipeline([('features', build_text_features()), ('clf', LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE))]),
    'svc_tuned': Pipeline([('features', build_text_features((2, 6), (1, 3), min_df=1)), ('clf', LinearSVC(C=0.25, max_iter=6000, random_state=RANDOM_STATE))]),
    'logreg': Pipeline([('features', build_text_features()), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', solver='saga', random_state=RANDOM_STATE))]),
}

for name, model in models_to_try.items():
    start = time.time()
    model.fit(train_split['Literal_norm'], train_split['chapter'])
    pred = model.predict(val_split['Literal_norm'])
    score = accuracy_score(val_split['chapter'], pred)
    chapter_models[name] = model
    print(f'{name}: {score:.4f} ({time.time() - start:.1f}s)')

svc_plain: 0.5427 (2.4s)
svc_tuned: 0.5438 (2.9s)
logreg: 0.5011 (178.8s)


d:\Users\herme\UNI\2\Natural language\project\project\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## 6. Reference-Augmented Chapter Model

This model expands the training signal with official ICD descriptions. The original training set has labelled clinical literals, while the reference table has many labelled code descriptions. By training on both, the model sees more vocabulary for each chapter and learns a better map from medical language to broad ICD category. This is especially useful for rare chapters and terms that are underrepresented in the training literals.

In [21]:
def train_reference_augmented_chapter_model(train_part, reference_df, c_value=1.0):
    # ICD descriptions act as extra labelled examples for the same first-character target.
    texts = pd.concat([train_part['Literal_norm'], reference_df['Description_norm']], ignore_index=True)
    labels = pd.concat([train_part['chapter'], reference_df['chapter']], ignore_index=True)
    model = Pipeline([
        ('features', build_text_features()),
        ('clf', LinearSVC(C=c_value, max_iter=3000, random_state=RANDOM_STATE)),
    ])
    model.fit(texts, labels)
    return model


start = time.time()
chapter_ref_model = train_reference_augmented_chapter_model(train_split, reference, c_value=1.0)
chapter_ref_pred = chapter_ref_model.predict(val_split['Literal_norm'])
chapter_ref_score = accuracy_score(val_split['chapter'], chapter_ref_pred)
print(f'reference-augmented chapter validation: {chapter_ref_score:.4f}')
print(f'done in {time.time() - start:.1f}s')

reference-augmented chapter validation: 0.5588
done in 126.1s


## 7. Dense SVD Embedding Model

This is a lightweight embedding-style baseline. The sparse TF-IDF representation is compressed into a smaller dense vector, which can capture broader co-occurrence patterns between related fragments and words. It is less exact than raw character n-grams, but it gives the ensemble a different view of the same literal: more semantic and compressed, less literal string matching.

In [22]:
svd_embedding_model = Pipeline([
    ('features', build_text_features((2, 6), (1, 3), min_df=1)),
    ('svd', TruncatedSVD(n_components=300, random_state=RANDOM_STATE)),
    ('norm', Normalizer(copy=False)),
    ('clf', LinearSVC(C=2.0, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE)),
])

start = time.time()
svd_embedding_model.fit(train_split['Literal_norm'], train_split['chapter'])
svd_pred = svd_embedding_model.predict(val_split['Literal_norm'])
print(f'SVD embedding chapter validation: {accuracy_score(val_split["chapter"], svd_pred):.4f}')
print(f'done in {time.time() - start:.1f}s')

SVD embedding chapter validation: 0.4682
done in 27.5s


## 8. Sentence Embeddings

Set `RUN_SENTENCE_EMBEDDINGS = True` in the setup cell to run this. In previous tests this was worse than TF-IDF for this task.

In [23]:
if RUN_SENTENCE_EMBEDDINGS:
    from sentence_transformers import SentenceTransformer

    model_name = 'intfloat/multilingual-e5-small'
    embedder = SentenceTransformer(model_name)
    x_train = embedder.encode(['query: ' + x for x in train_split['Literal_clean']], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    x_val = embedder.encode(['query: ' + x for x in val_split['Literal_clean']], batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    emb_clf = LinearSVC(C=4.0, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE)
    emb_clf.fit(x_train, train_split['chapter'])
    emb_pred = emb_clf.predict(x_val)
    print(f'sentence embedding chapter validation: {accuracy_score(val_split["chapter"], emb_pred):.4f}')
else:
    print('Skipped. Set RUN_SENTENCE_EMBEDDINGS = True to run this cell.')

Skipped. Set RUN_SENTENCE_EMBEDDINGS = True to run this cell.


## 9. Transformer Fine-Tuning

Set `RUN_TRANSFORMER_FINETUNE = True` in the setup cell to run this. CPU training is slow and may underperform without a better biomedical model or GPU.

In [24]:
if RUN_TRANSFORMER_FINETUNE:
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    model_name = 'mrm8488/TinyBERT-spanish-uncased-finetuned-ner'
    labels = sorted(train['chapter'].unique())
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels), ignore_mismatched_sizes=True)

    class ChapterDataset(Dataset):
        def __init__(self, df):
            self.text = df['Literal_clean'].fillna('').tolist()
            self.y = [label2id[x] for x in df['chapter']]
        def __len__(self):
            return len(self.text)
        def __getitem__(self, idx):
            enc = tokenizer(self.text[idx], truncation=True, max_length=48, padding='max_length', return_tensors='pt')
            item = {k: v.squeeze(0) for k, v in enc.items()}
            item['labels'] = torch.tensor(self.y[idx])
            return item

    train_loader = DataLoader(ChapterDataset(train_split), batch_size=32, shuffle=True)
    val_loader = DataLoader(ChapterDataset(val_split), batch_size=64)
    optimizer = torch.optim.AdamW(transformer.parameters(), lr=3e-5)

    for epoch in range(3):
        transformer.train()
        for batch in train_loader:
            optimizer.zero_grad()
            loss = transformer(**batch).loss
            loss.backward()
            optimizer.step()
        transformer.eval()
        preds, truth = [], []
        with torch.no_grad():
            for batch in val_loader:
                y = batch.pop('labels')
                logits = transformer(**batch).logits
                preds.extend(logits.argmax(1).tolist())
                truth.extend(y.tolist())
        print(f'epoch {epoch + 1}: {accuracy_score(truth, preds):.4f}')
else:
    print('Skipped. Set RUN_TRANSFORMER_FINETUNE = True to run this cell.')

Skipped. Set RUN_TRANSFORMER_FINETUNE = True to run this cell.


## 10. Full-Code ML Classifier

This model tries to predict the complete ICD code, not just the chapter. It is much harder because there are thousands of possible labels and many codes have only a few examples. Even when the full code is wrong, its first character can still be useful as another chapter prediction signal for the ensemble.

In [25]:
def filter_by_min_examples(df, min_count=2):
    counts = df['Code'].value_counts()
    return df[df['Code'].isin(counts[counts >= min_count].index)].copy()


full_train = filter_by_min_examples(train_split, min_count=2)
full_code_model = Pipeline([
    ('features', build_text_features((3, 5), (1, 2), min_df=2)),
    ('clf', LinearSVC(C=2.0, class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE)),
])

start = time.time()
full_code_model.fit(full_train['Literal_norm'], full_train['Code'])
full_code_pred = full_code_model.predict(val_split['Literal_norm'])
print(f'full-code classifier validation: {accuracy_score(val_split["Code"], full_code_pred):.4f}')
print(f'full-code classifier chapter validation: {accuracy_score(val_split["chapter"], pd.Series(full_code_pred).str[0]):.4f}')
print(f'done in {time.time() - start:.1f}s')

full-code classifier validation: 0.3277
full-code classifier chapter validation: 0.4442
done in 42.5s


## 11. Chapter Meta-Classifier Ensemble

This is the decision layer. Each base model gives its own chapter prediction, and the meta-classifier learns when to trust each one. For example, matching may be best for repeated literals, retrieval may help on dictionary-like phrases, and the reference-augmented classifier may be best on broader medical wording. The ensemble uses only model outputs and confidence/agreement signals; it does not use the previous Kaggle submission.

In [26]:
def build_meta_input(base_df, columns=None):
    # The meta-classifier sees model predictions as categorical features plus confidence signals.
    categorical_cols = [
        'match_chapter',
        'retrieval_chapter',
        'svc_plain',
        'svc_tuned',
        'logreg',
        'reference_augmented',
        'svd_embedding',
        'full_code_chapter',
        'majority_vote',
    ]
    numeric_cols = ['match_score', 'retrieval_score', 'reference_margin', 'agreement_count']

    cat = pd.get_dummies(base_df[categorical_cols].fillna(''), prefix=categorical_cols)
    num = base_df[numeric_cols].fillna(0).astype(float)
    features = pd.concat([cat, num], axis=1)

    if columns is not None:
        # Keep train/test one-hot columns aligned, even if a category appears only on one side.
        features = features.reindex(columns=columns, fill_value=0)
    return features


def majority_vote_row(row, cols):
    votes = [row[col] for col in cols if pd.notna(row[col]) and row[col] != '']
    if not votes:
        return ''
    return Counter(votes).most_common(1)[0][0]


def agreement_count_row(row, cols):
    votes = [row[col] for col in cols if pd.notna(row[col]) and row[col] != '']
    if not votes:
        return 0
    return Counter(votes).most_common(1)[0][1]


base_val = val_split[['chapter']].copy()
base_val['match_chapter'] = val_match['fuzzy_pred'].fillna('').values
base_val['match_score'] = val_match['fuzzy_score'].fillna(0).values
base_val['retrieval_chapter'] = val_ret['retrieval_chapter'].fillna('').values
base_val['retrieval_score'] = val_ret['retrieval_score'].fillna(0).values
base_val['svc_plain'] = chapter_models['svc_plain'].predict(val_split['Literal_norm'])
base_val['svc_tuned'] = chapter_models['svc_tuned'].predict(val_split['Literal_norm'])
base_val['logreg'] = chapter_models['logreg'].predict(val_split['Literal_norm'])
base_val['reference_augmented'] = chapter_ref_pred
base_val['svd_embedding'] = svd_pred
base_val['full_code_chapter'] = pd.Series(full_code_pred, index=val_split.index).astype(str).str[0].values

reference_scores = chapter_ref_model.decision_function(val_split['Literal_norm'])
base_val['reference_margin'] = reference_scores.max(axis=1)

vote_cols = [
    'match_chapter',
    'retrieval_chapter',
    'svc_plain',
    'svc_tuned',
    'logreg',
    'reference_augmented',
    'svd_embedding',
    'full_code_chapter',
]
base_val['majority_vote'] = base_val.apply(lambda row: majority_vote_row(row, vote_cols), axis=1)
base_val['agreement_count'] = base_val.apply(lambda row: agreement_count_row(row, vote_cols), axis=1)

# Hold out part of the validation predictions to check if stacking helps beyond voting.
meta_train, meta_holdout = train_test_split(
    base_val,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=base_val['chapter'],
)

x_meta_train = build_meta_input(meta_train)
meta_columns = x_meta_train.columns
x_meta_holdout = build_meta_input(meta_holdout, columns=meta_columns)

meta_classifier = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver='lbfgs',
    random_state=RANDOM_STATE,
)
meta_classifier.fit(x_meta_train, meta_train['chapter'])

meta_holdout_pred = meta_classifier.predict(x_meta_holdout)
print(f'meta-classifier holdout chapter accuracy: {accuracy_score(meta_holdout["chapter"], meta_holdout_pred):.4f}')
print(f'majority-vote holdout chapter accuracy: {accuracy_score(meta_holdout["chapter"], meta_holdout["majority_vote"]):.4f}')
print(f'reference model holdout chapter accuracy: {accuracy_score(meta_holdout["chapter"], meta_holdout["reference_augmented"]):.4f}')

# Refit on all validation-derived base predictions before predicting leaderboard rows.
x_meta_all = build_meta_input(base_val)
meta_columns = x_meta_all.columns
meta_classifier.fit(x_meta_all, base_val['chapter'])
base_val.head()

d:\Users\herme\UNI\2\Natural language\project\project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


meta-classifier holdout chapter accuracy: 0.5117
majority-vote holdout chapter accuracy: 0.5511
reference model holdout chapter accuracy: 0.5599


d:\Users\herme\UNI\2\Natural language\project\project\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,chapter,match_chapter,match_score,retrieval_chapter,retrieval_score,svc_plain,svc_tuned,logreg,reference_augmented,svd_embedding,full_code_chapter,reference_margin,majority_vote,agreement_count
10334,6,N,100.000000,N,0.643352,N,N,N,N,N,N,0.930679,N,8
10856,O,O,100.000000,L,0.937437,O,O,L,L,L,L,0.022708,L,5
1261,B,8,100.000000,P,0.139579,8,8,8,8,B,8,0.610823,8,6
8839,D,D,100.000000,D,0.354856,D,D,D,D,2,D,0.904748,D,7
7981,T,0,86.666667,R,0.615640,0,0,5,5,0,0,-0.189706,0,5


## 12. Train Final Base Models On All Data And Predict Test Chapters

After evaluating the approach on a validation split, the base models are trained again using all available labelled training data. This gives each model the strongest possible training signal before predicting the leaderboard rows. Their predictions are then passed to the meta-classifier to produce the final chapter/category output.

In [27]:
final_chapter_models = {}
for name, template in models_to_try.items():
    model = template
    model.fit(train['Literal_norm'], train['chapter'])
    final_chapter_models[name] = model

final_reference_model = train_reference_augmented_chapter_model(train, reference, c_value=1.0)
final_svd_model = Pipeline([
    ('features', build_text_features((2, 6), (1, 3), min_df=1)),
    ('svd', TruncatedSVD(n_components=300, random_state=RANDOM_STATE)),
    ('norm', Normalizer(copy=False)),
    ('clf', LinearSVC(C=2.0, class_weight='balanced', max_iter=5000, random_state=RANDOM_STATE)),
])
final_svd_model.fit(train['Literal_norm'], train['chapter'])

final_full_train = filter_by_min_examples(train, min_count=2)
final_full_model = Pipeline([
    ('features', build_text_features((3, 5), (1, 2), min_df=2)),
    ('clf', LinearSVC(C=2.0, class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE)),
])
final_full_model.fit(final_full_train['Literal_norm'], final_full_train['Code'])

# Recreate every base model prediction for the leaderboard rows, then stack them.
test_match_chapter = exact_fuzzy_predict(train, test, label_col='chapter', threshold=60)
test_ret = retrieve_best(test, reference, retrieval_index)
test_reference_scores = final_reference_model.decision_function(test['Literal_norm'])
test_full_code_pred = final_full_model.predict(test['Literal_norm'])

base_test = test[['id', 'Literal', 'Literal_norm']].copy()
base_test['match_chapter'] = test_match_chapter['fuzzy_pred'].fillna('').values
base_test['match_score'] = test_match_chapter['fuzzy_score'].fillna(0).values
base_test['retrieval_chapter'] = test_ret['retrieval_chapter'].fillna('').values
base_test['retrieval_score'] = test_ret['retrieval_score'].fillna(0).values
base_test['svc_plain'] = final_chapter_models['svc_plain'].predict(test['Literal_norm'])
base_test['svc_tuned'] = final_chapter_models['svc_tuned'].predict(test['Literal_norm'])
base_test['logreg'] = final_chapter_models['logreg'].predict(test['Literal_norm'])
base_test['reference_augmented'] = final_reference_model.predict(test['Literal_norm'])
base_test['svd_embedding'] = final_svd_model.predict(test['Literal_norm'])
base_test['full_code_chapter'] = pd.Series(test_full_code_pred).astype(str).str[0].values
base_test['reference_margin'] = test_reference_scores.max(axis=1)
base_test['majority_vote'] = base_test.apply(lambda row: majority_vote_row(row, vote_cols), axis=1)
base_test['agreement_count'] = base_test.apply(lambda row: agreement_count_row(row, vote_cols), axis=1)

x_meta_test = build_meta_input(base_test, columns=meta_columns)
base_test['y_category'] = meta_classifier.predict(x_meta_test)

print(base_test['y_category'].value_counts().head(15))
display(base_test.head())

d:\Users\herme\UNI\2\Natural language\project\project\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


y_category
Z    677
O    590
0    552
N    328
E    325
B    309
V    309
K    285
D    244
I    244
F    236
R    223
9    206
M    189
6    186
Name: count, dtype: int64


,id,Literal,Literal_norm,match_chapter,match_score,retrieval_chapter,retrieval_score,svc_plain,svc_tuned,logreg,reference_augmented,svd_embedding,full_code_chapter,reference_margin,majority_vote,agreement_count,y_category
0,1,AMNIODRENAJE,amniodrenaje,1,100.000000,1,0.173946,7,1,1,7,1,7,-0.082909,1,5,1
1,2,Hiperparatiroidismo primario,hiperparatiroidismo primario,E,100.000000,E,1.000000,E,E,E,E,E,E,0.522081,E,8,E
2,3,MIGRANYA parto,migranya parto,G,96.296296,G,0.347332,O,G,O,O,G,G,0.250330,G,5,G
3,4,VHC,vhc,B,100.000000,A,0.000000,B,B,B,B,B,B,0.373821,B,7,B
4,5,Absceso mama izq,absceso mama izq,6,85.714286,N,0.639651,N,N,N,N,N,6,0.388206,N,6,N


## 13. Save Submissions

This saves the meta-classifier output and one separate Kaggle-ready chapter submission for each individual model. These separate files are useful for Kaggle testing and for seeing which model contributes useful predictions.

In [ ]:
chapter_submission = base_test[['id', 'Literal', 'y_category']].copy()
chapter_submission.to_csv(SUBMISSIONS_DIR / 'notebook_meta_chapter_submission.csv', index=False)
base_test.to_csv(PREDICTIONS_DIR / 'notebook_meta_chapter_predictions.csv', index=False)

individual_submission_cols = {
    'match_chapter': 'notebook_model_match_chapter_submission.csv',
    'retrieval_chapter': 'notebook_model_retrieval_chapter_submission.csv',
    'svc_plain': 'notebook_model_svc_plain_submission.csv',
    'svc_tuned': 'notebook_model_svc_tuned_submission.csv',
    'logreg': 'notebook_model_logreg_submission.csv',
    'reference_augmented': 'notebook_model_reference_augmented_submission.csv',
    'svd_embedding': 'notebook_model_svd_embedding_submission.csv',
    'full_code_chapter': 'notebook_model_full_code_chapter_submission.csv',
    'majority_vote': 'notebook_model_majority_vote_submission.csv',
}

saved_files = [SUBMISSIONS_DIR / 'notebook_meta_chapter_submission.csv']
for col, filename in individual_submission_cols.items():
    submission = base_test[['id', 'Literal']].copy()
    submission['y_category'] = base_test[col].fillna('null').astype(str)
    submission.loc[submission['y_category'] == '', 'y_category'] = 'null'
    path = SUBMISSIONS_DIR / filename
    submission.to_csv(path, index=False)
    saved_files.append(path)

print('Saved submissions:')
for path in saved_files:
    print(' ', path)
print()
print('Meta submission shape:', chapter_submission.shape)
print('Columns:', list(chapter_submission.columns))


Saved submissions:
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_meta_chapter_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_match_chapter_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_retrieval_chapter_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_svc_plain_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_svc_tuned_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_logreg_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_reference_augmented_submission.csv
  d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\notebook_model_svd_embedding_submission.csv
  d:\Users\herme\UNI\2\Natural language\p